In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Model

def classification_model():

    model = models.Sequential([

        layers.Conv2D(
            32,
            (3,3),
            activation='relu',
            input_shape=(128,128,3)
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            64,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            128,
            (3,3),
            activation='relu'
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            128,
            activation='relu'
        ),

        layers.Dense(
            4,
            activation='softmax'
        )

    ])

    return model


# Build

model = classification_model()


# Optimizer

optimizer = tf.keras.optimizers.Adam(1e-4)


# Data

x_train = np.random.rand(
    500,
    128,
    128,
    3
)

y_train = tf.keras.utils.to_categorical(
    np.random.randint(0,4,500),
    num_classes=4
)


# Compile

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# Train

model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=16
)


# Fisher

fisher_matrix = {}

for var in model.trainable_variables:

    fisher_matrix[var.name] = tf.zeros_like(var)


# Dataset

dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
).batch(1)


# Gradients

for x_batch, y_batch in dataset:

    with tf.GradientTape() as tape:

        predictions = model(x_batch)

        loss = tf.keras.losses.categorical_crossentropy(
            y_batch,
            predictions
        )

    grads = tape.gradient(
        loss,
        model.trainable_variables
    )

    for var, grad in zip(
        model.trainable_variables,
        grads
    ):

        fisher_matrix[var.name] += tf.square(grad)


# Average

for var in model.trainable_variables:

    fisher_matrix[var.name] /= len(x_train)


# Weights

old_weights = {}

for var in model.trainable_variables:

    old_weights[var.name] = var.numpy()


# Save

model.save("phase1_classification.h5")



In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Model

def segmentation_model():

    inputs = layers.Input(
        shape=(128,128,3)
    )

    c1 = layers.Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(inputs)

    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(p1)

    p2 = layers.MaxPooling2D()(c2)

    b1 = layers.Conv2D(
        128,
        3,
        activation='relu',
        padding='same'
    )(p2)

    u1 = layers.Conv2DTranspose(
        64,
        2,
        strides=2,
        padding='same'
    )(b1)

    u1 = layers.concatenate([u1,c2])

    c3 = layers.Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(u1)

    u2 = layers.Conv2DTranspose(
        32,
        2,
        strides=2,
        padding='same'
    )(c3)

    u2 = layers.concatenate([u2,c1])

    c4 = layers.Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(u2)

    outputs = layers.Conv2D(
        1,
        1,
        activation='sigmoid'
    )(c4)

    model = models.Model(
        inputs,
        outputs
    )

    return model


# Build

model = segmentation_model()


# Optimizer

optimizer = tf.keras.optimizers.Adam(1e-4)


# Lambda

LAMBDA_EWC = 1000


# Loss

def ewc_loss(y_true, y_pred):

    base_loss = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(
            y_true,
            y_pred
        )
    )

    ewc_penalty = 0

    for var in model.trainable_variables:

        if var.name in fisher_matrix:

            ewc_penalty += tf.reduce_sum(
                fisher_matrix[var.name] *
                tf.square(
                    var - old_weights[var.name]
                )
            )

    total_loss = base_loss + (
        LAMBDA_EWC * ewc_penalty
    )

    return total_loss


# Data

x_seg = np.random.rand(
    300,
    128,
    128,
    3
)

y_seg = np.random.randint(
    0,
    2,
    (300,128,128,1)
)


# Compile

model.compile(
    optimizer=optimizer,
    loss=ewc_loss,
    metrics=['accuracy']
)


# Train

model.fit(
    x_seg,
    y_seg,
    epochs=10,
    batch_size=8
)


# Save

model.save("phase2_segmentation.h5")


In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np


# Model

def prediction_model():

    model = models.Sequential([

        layers.Conv2D(
            32,
            (3,3),
            activation='relu',
            input_shape=(128,128,3)
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            64,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            128,
            (3,3),
            activation='relu'
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            128,
            activation='relu'
        ),

        layers.Dense(
            1,
            activation='linear'
        )

    ])

    return model


# Build

model = prediction_model()


# Optimizer

optimizer = tf.keras.optimizers.Adam(1e-4)


# Lambda

LAMBDA_EWC = 1000


# Loss

def ewc_loss(y_true, y_pred):

    base_loss = tf.reduce_mean(
        tf.keras.losses.mse(
            y_true,
            y_pred
        )
    )

    ewc_penalty = 0

    for var in model.trainable_variables:

        if var.name in fisher_matrix:

            ewc_penalty += tf.reduce_sum(
                fisher_matrix[var.name] *
                tf.square(
                    var - old_weights[var.name]
                )
            )

    total_loss = base_loss + (
        LAMBDA_EWC * ewc_penalty
    )

    return total_loss


# Data

x_pred = np.random.rand(
    300,
    128,
    128,
    3
)

y_pred = np.random.rand(
    300,
    1
)


# Compile

model.compile(
    optimizer=optimizer,
    loss=ewc_loss,
    metrics=['mae']
)


# Train

model.fit(
    x_pred,
    y_pred,
    epochs=10,
    batch_size=16
)


# Save

model.save("phase3_prediction.h5")
